<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# H13's Proof of Concept on Synthetic Worms

A proof of concept of a method, not a brain: synthetic worms with a known hidden brain of five modules (one a decoy the
truth does not use), a modular brain of the same kinds trained as a whole against their behaviour statistics, then
ground down by ablation and pruning, with a held-out group. The question, the method and the four criteria are in the
journal of 2026-10-17, *H13's Proof of Concept on Synthetic Worms*, written before this ran. No real worm is used: the
benchmark is not frozen. numpy only; deterministic, every random number seeded.

## The synthetic worm

Five modules - S (roam/dwell switch), R (state-dependent spontaneous reversals), U (undulation whose frequency rises with speed), H (a tap reflex that habituates), X (a decoy: reversals before an expected tap, zero in the truth) - and the 12 statistics read from observables only.

In [1]:
import numpy as np
DT, T, TAP0, TAPDT, NTAP = 0.2, 600.0, 300.0, 10.0, 30
STEPS = int(T / DT)
NAMES = ["k_rd", "k_dr", "h_roam", "h_dwell", "d_rev", "v_roam", "v_dwell", "v_rev", "f0", "f1", "p0", "dep", "tau", "g"]
MODULES = {"S": ["k_rd", "k_dr"], "R": ["h_roam", "h_dwell"], "U": ["f1"], "H": ["dep"], "X": ["g"]}
BASELINE = {"k_rd": 0.0, "k_dr": 0.0, "h_roam": 0.0, "h_dwell": 0.0, "f1": 0.0, "dep": 0.0, "g": 0.0}
TRUTH = dict(k_rd=1/60, k_dr=1/40, h_roam=1/40, h_dwell=1/8, d_rev=2.0, v_roam=0.20, v_dwell=0.04, v_rev=0.12,
             f0=0.25, f1=1.5, p0=0.8, dep=0.12, tau=60.0, g=0.0)

def simulate(P, rng, reps=1):
    """P: dict name -> array (batch,) of parameters, one simulated individual per entry. Returns observables. With
    reps > 1 the batch is reps candidates of B // reps individuals each, candidate-major, and every candidate gets the
    same random numbers (common random numbers)."""
    B = len(P["k_rd"]); B0 = B // reps
    draw = lambda *shape: np.tile(rng.random(shape + (B0,)), (1,) * len(shape) + (reps,))
    roam = draw() < P["k_dr"] / np.maximum(P["k_rd"] + P["k_dr"], 1e-12)
    roam |= (P["k_rd"] + P["k_dr"]) == 0
    rev_left = np.zeros(B); w = np.ones(B); phase = draw() * 2 * np.pi
    speed = np.zeros((STEPS, B)); bend = np.zeros((STEPS, B)); onset = np.zeros((STEPS, B), bool)
    tap_steps = set(int(round((TAP0 + i * TAPDT) / DT)) for i in range(NTAP))
    for t in range(STEPS):
        u = draw(4)
        flip = np.where(roam, u[0] < P["k_rd"] * DT, u[0] < P["k_dr"] * DT)
        roam = roam ^ flip
        tsec = t * DT
        since = (tsec - TAP0) % TAPDT if tsec >= TAP0 - TAPDT else 0.0
        anticip = P["g"] * ((tsec >= TAP0 - TAPDT) & (since > TAPDT - 2.0) & (tsec < TAP0 + NTAP * TAPDT))
        h = np.where(roam, P["h_roam"], P["h_dwell"]) + anticip
        fwd = rev_left <= 0
        start = fwd & (u[1] < h * DT)
        if t in tap_steps:
            start |= fwd & (u[2] < P["p0"] * w)
            w = w * (1 - P["dep"])
        w = w + (1 - w) * DT / P["tau"]
        rev_left = np.where(start, P["d_rev"] * np.tile(rng.standard_exponential(B0), reps), rev_left - DT)
        rev = rev_left > 0
        onset[t] = start
        v = np.where(rev, -P["v_rev"], np.where(roam, P["v_roam"], P["v_dwell"])) * (1 + 0.1 * u[3] - 0.05)
        speed[t] = v
        phase = phase + 2 * np.pi * (P["f0"] + P["f1"] * np.abs(v)) * DT * np.where(rev, -1, 1)
        bend[t] = np.sin(phase)
    return speed, bend, onset

def stats(speed, bend, _onset=None):
    """Summary statistics from observables only: the speed and the bend; reversal onsets are read from the speed's
    sign (the simulator's own onset flag is ignored)."""
    S, B = speed.shape
    neg = speed < 0
    onset = neg & ~np.vstack([np.zeros((1, B), bool), neg[:-1]])
    win = int(10 / DT); nw = S // win
    a = np.abs(speed[: nw * win]).reshape(nw, win, B).mean(1)
    high = a > 0.1
    out = []
    out.append(high.mean(0))                                                           # share of fast windows
    out.append(np.where(high.any(0), (a * high).sum(0) / np.maximum(high.sum(0), 1), 0))  # speed in fast windows
    out.append(np.where((~high).any(0), (a * ~high).sum(0) / np.maximum((~high).sum(0), 1), 0))
    pre = int(TAP0 / DT) - int(10 / DT)
    out.append(onset[:pre].sum(0) / (pre * DT / 60))                                     # spontaneous reversals per min
    rev = speed < 0
    starts = onset.sum(0)
    out.append(rev.sum(0) * DT / np.maximum(starts, 1))                                  # mean reversal duration
    taps = [int(round((TAP0 + i * TAPDT) / DT)) for i in range(NTAP)]
    resp = np.array([onset[k:k + int(1 / DT)].any(0) for k in taps])                     # NTAP x B
    for lo, hi in [(0, 3), (3, 10), (10, 20), (20, 30)]:
        out.append(resp[lo:hi].mean(0))
    zc = (np.diff(np.sign(bend), axis=0) != 0)[: nw * win - 1]
    zcw = np.concatenate([zc, np.zeros((1, B), bool)]).reshape(nw, win, B).sum(1) / (2 * 10)
    out.append(np.where(high.any(0), (zcw * high).sum(0) / np.maximum(high.sum(0), 1), 0))   # bend frequency, fast
    out.append(np.where((~high).any(0), (zcw * ~high).sum(0) / np.maximum((~high).sum(0), 1), 0))
    ant = np.zeros(S, bool)
    for k in taps: ant[max(k - int(2 / DT), 0):k] = True
    tapzone = np.zeros(S, bool); tapzone[taps[0] - int(10 / DT): taps[-1]] = True
    other = tapzone & ~ant
    out.append(onset[ant].sum(0) / (ant.sum() * DT / 60) - onset[other].sum(0) / (other.sum() * DT / 60))  # anticipation
    return np.array(out).T                                                                # B x 12

## Groups, distance, scoring and the fit

In [2]:
GENS = 120
CV = 0.15
ACTIVE = [n for n in NAMES if n != "g"]

def individuals(mean, n, rng, reps=1):
    """n individuals around a mean parameter set: each positive parameter log-normal with a 15% CV; g as given. With
    reps > 1, the same n // reps variations for each of reps candidates (common random numbers)."""
    sd = np.sqrt(np.log(1 + CV ** 2))
    P = {}
    for k in NAMES:
        m = np.broadcast_to(np.asarray(mean[k], float), (n,)) if np.ndim(mean[k]) == 0 else np.asarray(mean[k], float)
        P[k] = m if k == "g" else m * np.exp(np.tile(rng.normal(-sd ** 2 / 2, sd, n // reps), reps))
    return P

def group_stats(mean, n, seed):
    rng = np.random.default_rng(seed)
    return stats(*simulate(individuals(mean, n, rng), rng))

def distance(model_mean_stats, target):
    sd = target.std(0, ddof=1)
    return float(np.mean(((model_mean_stats - target.mean(0)) / sd) ** 2))

def score(params, target, seeds=(101, 102, 103), n=64, offsets=None):
    ds = []
    for sd in seeds:
        m = group_stats(params, n, sd).mean(0)
        if offsets is not None: m = m + offsets
        ds.append(distance(m, target))
    return float(np.mean(ds)), float(np.std(ds, ddof=1))

def to_params(theta, pinned):
    p = {k: float(np.exp(theta[i])) for i, k in enumerate(NAMES)}
    p.update(pinned)
    return p

def fit(target, start, pinned, gens=GENS, lam=24, mu=6, nind=16, seed=7):
    rng = np.random.default_rng(seed)
    mean = np.array([np.log(start[k]) for k in NAMES]); spread = np.full(len(NAMES), 0.5)
    history = []
    for gen in range(gens):
        cand = mean + spread * rng.standard_normal((lam, len(NAMES)))
        cand[:, NAMES.index("g")] = np.clip(cand[:, NAMES.index("g")], -12, 3)
        batch = {k: np.repeat([to_params(c, pinned)[k] for c in cand], nind) for k in NAMES}
        r2 = np.random.default_rng(1000 + gen)                       # common random numbers within a generation
        sim = stats(*simulate(individuals(batch, lam * nind, r2, reps=lam), r2, reps=lam)).reshape(lam, nind, -1).mean(1)
        d = np.array([distance(s, target) for s in sim])
        elite = cand[np.argsort(d)[:mu]]
        mean = 0.7 * elite.mean(0) + 0.3 * mean
        spread = np.maximum(0.7 * elite.std(0) + 0.3 * spread, 0.02)
        history.append(float(d.min()))
    return to_params(mean, pinned), history

## The population and the full fit (criterion 1)

In [3]:
TRUTH_POP = individuals(TRUTH, 60, np.random.default_rng(2026))
rng = np.random.default_rng(2027)
pop_stats = stats(*simulate(TRUTH_POP, rng))
DEV, HELD = pop_stats[:30], pop_stats[30:]
srng = np.random.default_rng(2028)
START = {k: (0.2 if k == "g" else TRUTH[k] * float(np.exp(srng.normal(0, 0.5)))) for k in NAMES}

full, hist = fit(DEV, START, {})

print("D on dev by generation (every 20th):", [round(h, 3) for h in hist[::20]], round(hist[-1], 3))
err = {k: abs(np.log(full[k] / TRUTH[k])) for k in ACTIVE}
print("recovery |ln(fit/truth)|:", {k: round(float(v), 3) for k, v in err.items()})
print("median", round(float(np.median(list(err.values()))), 3), "max", round(max(err.values()), 3), "g fitted", round(full["g"], 4))
print("fit / truth:", {k: round(full[k] / TRUTH[k], 2) for k in ACTIVE})
d_full, s_full = score(full, HELD); print("held-out D full", round(d_full, 3), "scoring s", round(s_full, 3))
d_truth, s_truth = score(TRUTH, HELD); print("held-out D of the truth itself", round(d_truth, 3), round(s_truth, 3))

D on dev by generation (every 20th): [1.244, 0.281, 0.302, 0.22, 0.116, 0.081] 0.077
recovery |ln(fit/truth)|: {'k_rd': 0.637, 'k_dr': 0.019, 'h_roam': 0.619, 'h_dwell': 0.506, 'd_rev': 0.014, 'v_roam': 0.081, 'v_dwell': 0.065, 'v_rev': 0.878, 'f0': 0.015, 'f1': 0.013, 'p0': 0.016, 'dep': 0.081, 'tau': 0.382}
median 0.081 max 0.878 g fitted 0.0502
fit / truth: {'k_rd': 1.89, 'k_dr': 0.98, 'h_roam': 0.54, 'h_dwell': 0.6, 'd_rev': 1.01, 'v_roam': 0.92, 'v_dwell': 0.94, 'v_rev': 2.41, 'f0': 1.02, 'f1': 0.99, 'p0': 1.02, 'dep': 1.08, 'tau': 0.68}


held-out D full 0.166 scoring s 0.048


held-out D of the truth itself 0.036 0.011


## Ablation (criterion 2)

In [4]:
for mod, names in MODULES.items():
    p = dict(full); p.update({k: BASELINE[k] for k in names})
    d, s = score(p, HELD); print(f"ablate {mod}: held-out D {d:.3f} (s {s:.3f}), delta {d - d_full:+.3f}")

ablate S: held-out D 13.580 (s 0.033), delta +13.414


ablate R: held-out D 1.034 (s 0.038), delta +0.869


ablate U: held-out D 2.451 (s 0.179), delta +2.285


ablate H: held-out D 0.580 (s 0.069), delta +0.414


ablate X: held-out D 0.130 (s 0.034), delta -0.036


## Pruning the decoy (criterion 3)

In [5]:
pruned, hist2 = fit(DEV, START, {"g": 0.0})
d_pr, s_pr = score(pruned, HELD); print("pruned (no X): held-out D", round(d_pr, 3), "s", round(s_pr, 3), "minus full", round(d_pr - d_full, 3))

pruned (no X): held-out D 0.048 s 0.019 minus full -0.118


## The held-out catch (criterion 4)

In [6]:
DEV5 = DEV[:5]
mech5, _ = fit(DEV5, START, {})
m_dev, m_ds = score(mech5, DEV5); m_held, m_s = score(mech5, HELD)
base = np.mean([group_stats(START, 64, sd).mean(0) for sd in (101, 102, 103)], axis=0)
off = DEV5.mean(0) - base
f_dev, f_ds = score(START, DEV5, offsets=off); f_held, f_s = score(START, HELD, offsets=off)
print(f"scoring spreads: mechanistic dev {m_ds:.3f}, held {m_s:.3f}; fudge dev {f_ds:.3f}, held {f_s:.3f}")
print(f"mechanistic on 5: dev D {m_dev:.3f}, held D {m_held:.3f}, gap {m_held - m_dev:.3f}")
print(f"fudge on 5:       dev D {f_dev:.3f}, held D {f_held:.3f}, gap {f_held - f_dev:.3f}; gap difference {(f_held - f_dev) - (m_held - m_dev):.3f}")

scoring spreads: mechanistic dev 0.041, held 0.021; fudge dev 0.003, held 0.026
mechanistic on 5: dev D 0.224, held D 0.227, gap 0.004
fudge on 5:       dev D 0.013, held D 0.399, gap 0.386; gap difference 0.382
